In [1]:
!pip install sagemaker pandas scikit-learn

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import sagemaker

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
import boto3

s3 = boto3.client("s3")

response = s3.list_objects_v2(Bucket="parth-ml-model")
print(response)

{'ResponseMetadata': {'RequestId': 'D93407N8JHW7R1KX', 'HostId': 'PUN5IiReRW/90pwqTcjEhPy0y3zgTeDqy0bFrO+0clpeCB0sLfa5Yp0ndwyVlUL35rsbDRegaLXU5ZEAIvuCHDti2xSaV9Bw', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'PUN5IiReRW/90pwqTcjEhPy0y3zgTeDqy0bFrO+0clpeCB0sLfa5Yp0ndwyVlUL35rsbDRegaLXU5ZEAIvuCHDti2xSaV9Bw', 'x-amz-request-id': 'D93407N8JHW7R1KX', 'date': 'Mon, 16 Mar 2026 08:33:55 GMT', 'x-amz-bucket-region': 'ap-south-1', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'iris.csv', 'LastModified': datetime.datetime(2026, 3, 16, 8, 9, 32, tzinfo=tzlocal()), 'ETag': '"70d8697ffda45fd7f792a43d91ac0637"', 'ChecksumAlgorithm': ['CRC64NVME'], 'ChecksumType': 'FULL_OBJECT', 'Size': 4007, 'StorageClass': 'STANDARD'}], 'Name': 'parth-ml-model', 'Prefix': '', 'MaxKeys': 1000, 'EncodingType': 'url', 'KeyCount': 1}


In [4]:
import boto3
import pandas as pd

s3 = boto3.client("s3")

# download file from S3 to notebook
s3.download_file("parth-ml-model", "iris.csv", "iris.csv")

# read locally
data = pd.read_csv("iris.csv")

data.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [5]:
from sklearn.model_selection import train_test_split
X = data.drop("species", axis=1)
y = data["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
model = LogisticRegression(max_iter=200)

model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,200
,multi_class,'deprecated'


In [7]:
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [8]:
import joblib

joblib.dump(model, "model.joblib")

['model.joblib']

In [10]:
import boto3

s3 = boto3.client("s3")

s3.upload_file("model.joblib", "parth-ml-model", "model/model.joblib")

In [12]:
%%writefile inference.py
import joblib
import json
import numpy as np

def model_fn(model_dir):
    model = joblib.load(f"{model_dir}/model.joblib")
    return model

def input_fn(request_body, request_content_type):
    data = json.loads(request_body)
    return np.array(data)

def predict_fn(input_data, model):
    prediction = model.predict(input_data)
    return prediction.tolist()

def output_fn(prediction, content_type):
    return json.dumps(prediction)

Writing inference.py


In [15]:
import os

os.makedirs("model", exist_ok=True)

In [16]:
import shutil

shutil.move("model.joblib", "model/model.joblib")

'model/model.joblib'

In [17]:
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model", arcname=".")

In [18]:
import boto3

s3 = boto3.client("s3")

s3.upload_file("model.tar.gz", "parth-ml-model", "model/model.tar.gz")

In [19]:
from sagemaker.sklearn.model import SKLearnModel
import sagemaker

role = sagemaker.get_execution_role()

model = SKLearnModel(
    model_data="s3://parth-ml-model/model/model.tar.gz",
    role=role,
    entry_point="inference.py",
    framework_version="1.0-1"
)

predictor = model.deploy(
    instance_type="ml.t2.medium",
    initial_instance_count=1
)

------!

In [20]:
predictor.endpoint_name

'sagemaker-scikit-learn-2026-03-16-08-44-39-183'